# GraphRAG — Graph-Based Retrieval-Augmented Generation Demo

This notebook builds a minimal **GraphRAG** pipeline that runs **entirely locally**, except for the LLM calls, which use **Groq's `llama-3.3-70b-versatile`**.

Instead of retrieving raw text chunks by embedding similarity (as in `rag_demo.ipynb`), this pipeline extracts a **knowledge graph** of entities and relationships from the documents, then answers questions by retrieving a relevant **subgraph** and reasoning over its facts.

Pipeline:
1. **Load** a list of files from `data/` (`.txt`, `.md`, `.pdf`)
2. **Split** each file into overlapping chunks
3. **Extract** entities + relationships from each chunk with Groq (structured JSON output) and merge them into a single graph
4. **Store** the graph on disk (GraphML) and embed each entity node with a local sentence-transformer, storing embeddings in a local vector DB ([ChromaDB](https://www.trychroma.com/)) for semantic node lookup
5. **Visualize** the full knowledge graph as an interactive, force-directed network with node and edge labels
6. **Ask a question** → embed it, find the closest entity nodes, expand to their neighbors to build a subgraph
7. **Visualize the retrieved subgraph** with the matched entities highlighted
8. **Compile an answer** by sending the question + subgraph facts to Groq's Llama 3.3 70B model

Only Groq is used as an external API (for extraction + generation) — chunking, embeddings, the graph store, the vector DB, and both visualizations run locally.

## 0. Setup

This notebook expects to run under the `.venv` virtual environment in this project folder — select **Python (.venv)** as the kernel (top-right in VS Code / Jupyter) before running the cells below.

Dependencies are listed in `requirements.txt`; the cell below installs them into `.venv`. API keys are loaded from a local `.env` file via `python-dotenv` — copy `.env.example` to `.env` and fill in your Groq key (free tier at https://console.groq.com/keys):

```
GROQ_API_KEY=your-key-here
```

`.env` is never read by anything outside this notebook/process, so it keeps the key out of the notebook source itself.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os
import glob
import json
import time
import html as html_lib
from pathlib import Path

import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from dotenv import load_dotenv
from IPython.display import HTML, display

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from pypdf import PdfReader
from pyvis.network import Network

load_dotenv()

DATA_DIR = Path("data")
GRAPH_DB_DIR = Path("graph_db")           # where the graph + rendered HTML views are persisted
MODEL_CACHE_DIR = Path(".model_cache")    # local cache so the embedding model isn't re-downloaded each run
CHUNK_SIZE = 500        # characters per chunk
CHUNK_OVERLAP = 80      # overlap between consecutive chunks
EMBED_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"
GRAPH_CHUNK_LIMIT = 30   # number of chunks to run entity/relationship extraction on (keep the demo fast & inside free-tier rate limits — raise to cover the whole corpus)
TOP_K_NODES = 3          # number of seed entities to retrieve per question
HOPS = 1                 # how many relationship hops to expand around the seed entities

GRAPH_DB_DIR.mkdir(exist_ok=True)


## 1. Load documents

Drop `.txt`, `.md`, or `.pdf` files into the `data/` folder, then run the cell below to load them. (Reuses the same loader as `rag_demo.ipynb`.)

In [ ]:
DATA_DIR.mkdir(exist_ok=True)

def load_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

text_paths = sorted(glob.glob(str(DATA_DIR / "*.txt"))) + sorted(glob.glob(str(DATA_DIR / "*.md")))
pdf_paths = sorted(glob.glob(str(DATA_DIR / "*.pdf")))

documents = {}
for path in text_paths:
    documents[Path(path).name] = Path(path).read_text(encoding="utf-8", errors="ignore")
for path in pdf_paths:
    documents[Path(path).name] = load_pdf(path)

assert documents, f"No .txt/.md/.pdf files found in {DATA_DIR}/ — add some documents and re-run this cell."
print(f"Loaded {len(documents)} files: {list(documents.keys())}")


## 2. Split into chunks

Same fixed-size sliding-window splitter as `rag_demo.ipynb`. Each chunk keeps track of its source file so extracted facts can be traced back to a document.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

records = []  # each: {id, text, source}
for source, text in documents.items():
    for i, chunk in enumerate(chunk_text(text)):
        records.append({"id": f"{source}::{i}", "text": chunk, "source": source})

print(f"Created {len(records)} chunks from {len(documents)} documents")
pd.DataFrame(records)[["id", "source"]]


## 3. Extract a knowledge graph from each chunk

For each chunk, ask Groq's Llama 3.3 70B to extract entities (people, organizations, places, quantities, dates, etc.) and the relationships between them, as strict JSON. Results from all chunks are merged into a single `networkx.MultiDiGraph` — repeated relationships between the same pair of entities are collapsed into one edge with a running `weight` and the set of source chunks that mentioned it.

Only the first `GRAPH_CHUNK_LIMIT` chunks are processed by default to keep the demo fast and within Groq's free-tier rate limits — raise it to cover the whole corpus.

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found — copy .env.example to .env and add your key."
groq_client = Groq(api_key=groq_api_key)

EXTRACTION_SYSTEM_PROMPT = (
    "You are an information-extraction engine. Extract a knowledge graph (entities and "
    "relationships) from the given text chunk. Respond only with strict JSON, no prose."
)

def extract_graph_from_chunk(chunk_text, source):
    prompt = f"""Text chunk (source: {source}):
\"\"\"
{chunk_text}
\"\"\"

Extract:
1. "entities": a list of {{"name": ..., "type": ...}} — key people, organizations, countries, places, quantities, dates, projects, etc. mentioned in the text.
2. "relationships": a list of {{"source": ..., "target": ..., "relation": ...}} — a short verb-phrase relation connecting two entity names from the list above.

Respond as JSON: {{"entities": [...], "relationships": [...]}}"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

def add_relationship(G, source_entity, target_entity, relation, source_chunk_id):
    if G.has_edge(source_entity, target_entity, key=relation):
        G[source_entity][target_entity][relation]["weight"] += 1
        G[source_entity][target_entity][relation]["sources"].add(source_chunk_id)
    else:
        G.add_edge(source_entity, target_entity, key=relation, relation=relation, weight=1, sources={source_chunk_id})

G = nx.MultiDiGraph()
chunks_to_process = records[:GRAPH_CHUNK_LIMIT]
skipped = 0

for i, record in enumerate(chunks_to_process):
    try:
        graph_data = extract_graph_from_chunk(record["text"], record["source"])
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [skip] chunk {record['id']}: {e}")
        skipped += 1
        continue

    for entity in graph_data.get("entities", []):
        name = str(entity.get("name", "")).strip()
        if not name:
            continue
        if name not in G:
            G.add_node(name, type=entity.get("type", "unknown"), sources=set())
        G.nodes[name]["sources"].add(record["id"])

    for rel in graph_data.get("relationships", []):
        src = str(rel.get("source", "")).strip()
        tgt = str(rel.get("target", "")).strip()
        relation = str(rel.get("relation", "related to")).strip()
        if not src or not tgt:
            continue
        if src not in G:
            G.add_node(src, type="unknown", sources=set())
        if tgt not in G:
            G.add_node(tgt, type="unknown", sources=set())
        add_relationship(G, src, tgt, relation, record["id"])

    if (i + 1) % 10 == 0:
        print(f"  processed {i + 1}/{len(chunks_to_process)} chunks — {G.number_of_nodes()} nodes, {G.number_of_edges()} edges so far")
    time.sleep(0.3)  # gentle pacing against Groq's free-tier rate limits

print(f"\nBuilt graph from {len(chunks_to_process) - skipped}/{len(chunks_to_process)} chunks: "
      f"{G.number_of_nodes()} entities, {G.number_of_edges()} relationships")

print("\nFirst 10 relationships extracted:")
for u, v, data in list(G.edges(data=True))[:10]:
    print(f"  {u} --[{data.get('relation')}]--> {v}  (seen {data.get('weight', 1)}x)")


## 4. Store the graph and embed entity nodes

The graph itself is persisted to disk as GraphML (`graph_db/knowledge_graph.graphml`). Separately, each entity name is embedded with the same local multilingual sentence-transformer used in `rag_demo.ipynb` and stored in a local ChromaDB collection, so a question can be matched to the closest entity nodes by semantic similarity.

In [ ]:
# GraphML can't store Python sets, so export a copy with set attributes flattened to strings
G_export = G.copy()
for _, data in G_export.nodes(data=True):
    data["sources"] = "; ".join(sorted(data.get("sources", [])))
for _, _, data in G_export.edges(data=True):
    data["sources"] = "; ".join(sorted(data.get("sources", [])))

graphml_path = GRAPH_DB_DIR / "knowledge_graph.graphml"
nx.write_graphml(G_export, graphml_path)
print(f"Persisted graph to {graphml_path}")

embedder = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=str(MODEL_CACHE_DIR))

chroma_client = chromadb.PersistentClient(path="./chroma_db_graph")
collection = chroma_client.get_or_create_collection(
    name="graph_nodes",
    metadata={"hnsw:space": "cosine"},
)

# Reset collection contents so re-running the notebook doesn't duplicate nodes
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)

node_ids = list(G.nodes())
node_texts = [f"{n} ({G.nodes[n].get('type', 'unknown')})" for n in node_ids]
node_embeddings = embedder.encode(node_texts, show_progress_bar=True, normalize_embeddings=True)

collection.add(
    ids=node_ids,
    embeddings=[e.tolist() for e in node_embeddings],
    documents=node_texts,
    metadatas=[{"type": G.nodes[n].get("type", "unknown")} for n in node_ids],
)

print(f"Stored {collection.count()} entity nodes in the vector DB")


## 5. Visualize the full knowledge graph

Two views are produced:

- An **inline Plotly view** (below) — node labels are always visible; hover over a node or edge for its full detail. Renders directly in the notebook output, scroll to zoom, drag to pan.
- A **standalone [pyvis](https://pyvis.readthedocs.io/) HTML file**, saved to `graph_db/` — a draggable, physics-based force layout with node/edge labels. VS Code's notebook output sandbox doesn't run the `<script>` this needs to draw itself, so it can't be embedded inline — open the saved file directly in a browser tab for the fully interactive version.


In [ ]:
def build_plotly_graph(graph, highlight_nodes=None, title="Knowledge graph", height=650, seed=42):
    highlight_nodes = highlight_nodes or set()
    pos = nx.spring_layout(graph, seed=seed)

    edge_x, edge_y = [], []
    mid_x, mid_y, mid_text = [], [], []
    for u, v, data in graph.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        mid_x.append((x0 + x1) / 2)
        mid_y.append((y0 + y1) / 2)
        mid_text.append(f"{u} → {v}<br>{data.get('relation', '')} (seen {data.get('weight', 1)}x)")

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1, color="#666"),
        hoverinfo="none", showlegend=False,
    )
    # Invisible markers at edge midpoints so hovering anywhere along an edge shows its relation text
    edge_label_trace = go.Scatter(
        x=mid_x, y=mid_y, mode="markers",
        marker=dict(size=12, color="#666", opacity=0),
        hoverinfo="text", hovertext=mid_text, showlegend=False,
    )

    node_x, node_y, node_text, node_hover, node_color = [], [], [], [], []
    for node, data in graph.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
        sources = data.get("sources", set())
        n_mentions = len(sources) if isinstance(sources, set) else str(sources).count(";") + 1
        node_hover.append(f"{node}<br>type: {data.get('type', 'unknown')}<br>mentioned in {n_mentions} chunk(s)")
        node_color.append("#e74c3c" if node in highlight_nodes else "#3498db")

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=node_text, textposition="top center",
        textfont=dict(color="#e0e0e0", size=11),
        hoverinfo="text", hovertext=node_hover,
        marker=dict(size=16, color=node_color, line=dict(width=1, color="#1e1e1e")),
        showlegend=False,
    )

    fig = go.Figure(data=[edge_trace, edge_label_trace, node_trace])
    fig.update_layout(
        title=title,
        plot_bgcolor="#1e1e1e", paper_bgcolor="#1e1e1e",
        font=dict(color="#e0e0e0"),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        margin=dict(l=0, r=0, t=40, b=0),
        height=height,
    )
    return fig

def export_pyvis_html(graph, path, highlight_nodes=None, height="650px"):
    highlight_nodes = highlight_nodes or set()
    net = Network(height=height, width="100%", directed=True, notebook=True,
                   cdn_resources="in_line", bgcolor="#1e1e1e", font_color="#e0e0e0")
    net.barnes_hut()

    for node, data in graph.nodes(data=True):
        sources = data.get("sources", set())
        n_mentions = len(sources) if isinstance(sources, set) else str(sources).count(";") + 1
        title = f"{node}<br>type: {data.get('type', 'unknown')}<br>mentioned in {n_mentions} chunk(s)"
        color = "#e74c3c" if node in highlight_nodes else "#3498db"
        net.add_node(node, label=node, title=title, color=color)

    for u, v, data in graph.edges(data=True):
        relation = data.get("relation", "")
        title = f"{relation} (seen {data.get('weight', 1)}x)"
        net.add_edge(u, v, label=relation, title=title, arrows="to")

    # pyvis's own write_html() opens the file without an explicit encoding, which crashes on
    # non-Latin1 text (e.g. Hebrew) under Windows' default locale — write it ourselves instead.
    page = net.generate_html(notebook=True)
    out_path = Path(path).resolve()
    out_path.write_text(page, encoding="utf-8")
    return out_path

full_graph_path = export_pyvis_html(G, GRAPH_DB_DIR / "graph_full.html")
print(f"Draggable, physics-based view saved to: {full_graph_path}\n(open it directly in a browser tab)")

build_plotly_graph(G, title="Full knowledge graph").show()


## 6. Ask a question — retrieve a relevant subgraph

Embed the question with the same model used for the entity nodes, find the `TOP_K_NODES` closest entities, then expand `HOPS` relationship hops outward to pull in their connected facts.

In [ ]:
def retrieve_subgraph(question, k=TOP_K_NODES, hops=HOPS):
    q_embedding = embedder.encode([question], normalize_embeddings=True)[0]
    results = collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    seed_nodes = set(results["ids"][0])

    nodes_in_subgraph = set(seed_nodes)
    frontier = set(seed_nodes)
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            if node in G:
                next_frontier.update(G.predecessors(node))
                next_frontier.update(G.successors(node))
        nodes_in_subgraph.update(next_frontier)
        frontier = next_frontier

    subgraph = G.subgraph(nodes_in_subgraph).copy()
    return subgraph, seed_nodes, q_embedding

def subgraph_to_context(subgraph):
    lines = []
    for u, v, data in subgraph.edges(data=True):
        sources = data.get("sources", set())
        sources_str = ", ".join(sorted(sources)) if isinstance(sources, set) else str(sources)
        lines.append(f"{u} --[{data.get('relation')}]--> {v}  (source: {sources_str})")
    return "\n".join(lines)

question = "לאילו מדינות ישראל מייצאת גז ואילו כמויות?"
subgraph, seed_nodes, q_embedding = retrieve_subgraph(question)

print(f"Seed entities matched to the question: {seed_nodes}")
print(f"Retrieved subgraph: {subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges\n")
print(subgraph_to_context(subgraph))


### Bonus: visualize the retrieved subgraph

The matched seed entities are highlighted in red; entities pulled in via graph expansion are blue. As above, an inline Plotly view renders directly below, and a draggable pyvis file is saved to `graph_db/` for viewing in a browser.


In [ ]:
subgraph_path = export_pyvis_html(subgraph, GRAPH_DB_DIR / "graph_subgraph.html", highlight_nodes=seed_nodes, height="500px")
print(f"Draggable, physics-based view saved to: {subgraph_path}\n(open it directly in a browser tab)")

build_plotly_graph(subgraph, highlight_nodes=seed_nodes, title=f'Retrieved subgraph for: "{question}"', height=500).show()


## 7. Compile an answer with Groq (`llama-3.3-70b-versatile`)

The retrieved subgraph's edges are stitched into a list of `subject --[relation]--> object` facts and sent to the LLM along with the question. As in `rag_demo.ipynb`, the answer is rendered as right-to-left HTML in an output cell (via an embedded iframe) and saved to `answer.html`.

In [ ]:
def compile_answer_from_graph(question, subgraph):
    context = subgraph_to_context(subgraph)
    prompt = f"""Answer the question using only the knowledge-graph facts below. Each line is a
subject --[relation]--> object triple extracted from the source document(s). If the facts
don't contain the answer, say so.

Knowledge graph facts:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers strictly from the provided knowledge-graph facts and cites the source chunk(s) you used."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

def render_answer_html(question, answer, path="answer.html", height=420):
    page = f"""<!DOCTYPE html>
<html lang="he" dir="rtl">
<head>
<meta charset="utf-8">
<title>GraphRAG Answer</title>
<style>
    body {{
        font-family: "Segoe UI", Arial, sans-serif;
        margin: 20px;
        line-height: 1.7;
        background: #1e1e1e;
        color: #e0e0e0;
    }}
    h2 {{ font-size: 1rem; color: #999; margin-bottom: 4px; }}
    .question {{ font-size: 1.3rem; font-weight: bold; margin-bottom: 24px; color: #f5f5f5; }}
    .answer {{ font-size: 1.15rem; white-space: pre-wrap; color: #e0e0e0; }}
</style>
</head>
<body>
    <h2>שאלה</h2>
    <div class="question">{html_lib.escape(question)}</div>
    <h2>תשובה</h2>
    <div class="answer">{html_lib.escape(answer)}</div>
</body>
</html>
"""
    out_path = Path(path).resolve()
    out_path.write_text(page, encoding="utf-8")

    iframe = (
        f'<iframe srcdoc="{html_lib.escape(page)}" '
        f'style="width:100%; height:{height}px; border:1px solid #444; border-radius:6px;">'
        f"</iframe>"
    )
    display(HTML(iframe))
    return out_path

answer = compile_answer_from_graph(question, subgraph)
print(answer)

html_path = render_answer_html(question, answer)


## 8. End-to-end `ask()` helper

Combine subgraph retrieval + generation + both visualizations into a single function for interactive use.

In [ ]:
def ask(question, k=TOP_K_NODES, hops=HOPS, verbose=True):
    subgraph, seed_nodes, _ = retrieve_subgraph(question, k=k, hops=hops)
    answer = compile_answer_from_graph(question, subgraph)

    render_answer_html(question, answer)

    graph_path = export_pyvis_html(subgraph, GRAPH_DB_DIR / "graph_last_query.html", highlight_nodes=seed_nodes, height="450px")
    print(f"Draggable, physics-based view saved to: {graph_path}\n(open it directly in a browser tab)")
    build_plotly_graph(subgraph, highlight_nodes=seed_nodes, title=f'Retrieved subgraph for: "{question}"', height=450).show()

    if verbose:
        print(f"Seed entities: {seed_nodes}")
        print("Subgraph facts used:")
        print(subgraph_to_context(subgraph))
        print("\nAnswer:\n" + answer)
    return answer

# Try it yourself:
ask("לאילו מדינות ישראל מייצאת גז ואילו כמויות?")


## Next steps

- Raise `GRAPH_CHUNK_LIMIT` to build the graph from the whole corpus (slower, more Groq calls — watch your rate limits)
- Improve entity resolution — right now merging is exact-string-match, so "ישראל" and "מדינת ישראל" become separate nodes; try fuzzy matching or an LLM-based dedup pass
- Try `HOPS = 2` for broader context, or prune low-`weight` edges before sending facts to the LLM
- Tune `nx.spring_layout(...)` parameters (or swap in `nx.kamada_kawai_layout`) for a different inline layout; tune `net.set_options(...)` (pyvis) for the browser-viewed file
- Add a DOCX/HTML loader alongside the existing PDF/txt/md support for richer document sets
- Reload the graph from `graph_db/knowledge_graph.graphml` on startup instead of re-extracting every run
